# NB07 — Pipeline-artefact ablation and domain classifier

**Purpose.** Test how much of the apparent MIMIC→eICU shift is created by data-pipeline choices rather than by patients. Four feature variants are trained with identical hyper-parameters: V1 hourly-harmonised values (primary), V2 raw-resolution values (sampling-frequency artefact), V3 V1 + measurement counts (charting-intensity artefact), V4 V1 + missingness indicators. A domain classifier (MIMIC vs eICU) quantifies separability of each variant.

**Research question.** RQ3 — how much of 'dataset shift' is pipeline shift?

| | |
|---|---|
| **Inputs** | analysis parquet, `models/lgbm_best_params.json` |
| **Outputs** | Table 4 (ablation), Figure 4 |
| **Free-Colab runtime** | 20–30 min |

**How to run:** `Runtime → Run all`. If Colab disconnects, just run all again — finished steps are cached.
At the end, download the zip printed in the last cell into your local `Plan A/v4/results/` folder.

In [ ]:
# ==== Setup (identical in every notebook) ====
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
BASE = os.environ['TRANSPORT_BASE']
sys.path.insert(0, os.path.join(BASE, 'code'))
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
importlib.reload(tu)
warnings.filterwarnings('ignore')
tu.set_style()
OUT = tu.init_notebook('NB07')

In [ ]:
import json, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
m, e = tu.load_analysis()
bp = json.load(open(os.path.join(tu.MODELS, 'lgbm_best_params.json')))
m = tu.add_missing_indicators(m); e = tu.add_missing_indicators(e)
dev, tt = m[m['period'] == 'dev'], m[m['period'] == 'temporal_test']
def make_lgbm(n=None):
    return lgb.LGBMClassifier(n_estimators=n or bp['n_estimators'], subsample_freq=1, random_state=tu.SEED,
                              verbose=-1, n_jobs=-1, **bp['params'])
VARIANTS = {'V1 hourly values (primary)': 'V1', 'V2 raw-resolution values': 'V2',
            'V3 V1 + measurement counts': 'V3', 'V4 V1 + missingness indicators': 'V4'}

In [ ]:
def run_variant(label, v):
    F = tu.feature_set(v)
    mdl = make_lgbm().fit(dev[F].astype(float), dev['mortality'].astype(int))
    p_tt = mdl.predict_proba(tt[F].astype(float))[:, 1]
    p_e = mdl.predict_proba(e[F].astype(float))[:, 1]
    ci_t, _ = tu.bootstrap(tt['mortality'].values, p_tt, B=tu.n_boot(300),
                           metrics=['auroc', 'oe', 'citl', 'slope', 'ici'])
    ci_e, _ = tu.bootstrap(e['mortality'].values, p_e, clusters=e['hospital_id'].values, B=tu.n_boot(300),
                           metrics=['auroc', 'oe', 'citl', 'slope', 'ici'])
    ph = tu.per_hospital(e.assign(p=p_e), 'p', full=False)
    ma = tu.meta_analysis(ph.loc[ph['eligible'], 'ln_oe'], ph.loc[ph['eligible'], 'ln_oe_se'])
    rows = []
    for ds, ci in [('MIMIC temporal test', ci_t), ('eICU all', ci_e)]:
        for _, r in ci.iterrows():
            rows.append(dict(variant=label, dataset=ds, metric=r['metric'], estimate=r['estimate'], lo=r['lo'], hi=r['hi']))
    rows.append(dict(variant=label, dataset='eICU hospitals', metric='tau_ln_oe', estimate=ma['tau'], lo=np.nan, hi=np.nan))
    rows.append(dict(variant=label, dataset='eICU hospitals', metric='pct_acceptable_oe',
                     estimate=100 * ph.loc[ph['eligible'], 'oe'].between(*tu.OE_ACCEPT).mean(), lo=np.nan, hi=np.nan))
    return pd.DataFrame(rows)

abl = pd.concat([tu.cached(f'abl_{v}', run_variant, lab, v) for lab, v in VARIANTS.items()])
tu.save_table(abl.round(4), 'table4_ablation_long', show=False)
abl_w = abl.assign(txt=abl.apply(lambda r: tu.fmt_ci(r.estimate, r.lo, r.hi) if pd.notna(r.lo) else f'{r.estimate:.3f}', axis=1)) \
           .pivot_table(index=['dataset', 'metric'], columns='variant', values='txt', aggfunc='first')
tu.save_table(abl_w, 'table4_ablation', index=True)

### Domain classifier: how easily can a model tell MIMIC from eICU, per variant?

In [ ]:
rng = np.random.default_rng(tu.SEED)
ns = min(len(dev), len(e), 30000)
dm = pd.concat([dev.sample(ns, random_state=tu.SEED).assign(domain=0),
                e.sample(ns, random_state=tu.SEED).assign(domain=1)], ignore_index=True)
dom_rows, dom_imp = [], {}
for lab, v in VARIANTS.items():
    F = [f for f in tu.feature_set(v) if f != 'sex_male']
    clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1, random_state=tu.SEED)
    p = cross_val_predict(clf, dm[F].astype(float), dm['domain'], cv=StratifiedKFold(5, shuffle=True, random_state=tu.SEED),
                          method='predict_proba')[:, 1]
    dom_rows.append(dict(variant=lab, domain_auroc=roc_auc_score(dm['domain'], p)))
    clf.fit(dm[F].astype(float), dm['domain'])
    imp = pd.Series(clf.booster_.feature_importance('gain'), index=F)
    dom_imp[lab] = (100 * imp / imp.sum()).sort_values(ascending=False).head(10).round(1)
    if v == 'V1':
        dm['p_domain_v1'] = p
dom = pd.DataFrame(dom_rows)
tu.save_table(dom.round(4), 'domain_classifier_auroc')
for lab, s in dom_imp.items():
    print(f'\nTop domain-separating features -- {lab}:\n', s.to_string())
pd.DataFrame(dom_imp).to_csv(tu.out_path('domain_classifier_top_features.csv'))

In [ ]:
# Does a hospital's dissimilarity from MIMIC (V1 domain score) predict its miscalibration?
from scipy.stats import spearmanr
pred = pd.read_parquet(os.path.join(tu.MODELS, 'pred_eicu.parquet'))
ph = tu.per_hospital(pred, 'p_lgbm', full=False)
clf = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1, random_state=tu.SEED)
F1 = [f for f in tu.feature_set('V1') if f != 'sex_male']
clf.fit(dm[F1].astype(float), dm['domain'])
e['p_domain'] = clf.predict_proba(e[F1].astype(float))[:, 1]
hd = e.groupby('hospital_id')['p_domain'].mean().rename('dissimilarity').reset_index()
jj = ph[ph['eligible']].merge(hd, on='hospital_id')
rho_d, p_d = spearmanr(jj['dissimilarity'], jj['ln_oe'].abs())
print(f'Spearman(hospital dissimilarity, |ln O/E|) = {rho_d:.3f} (p={p_d:.2g}, k={len(jj)})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
labels = list(VARIANTS)
short = ['V1', 'V2', 'V3', 'V4']
for ax, met, ttl in zip(axes, ['oe', 'slope', 'auroc'], ['O/E in eICU', 'Calibration slope in eICU', 'AUROC in eICU']):
    d = abl[(abl['dataset'] == 'eICU all') & (abl['metric'] == met)].set_index('variant').loc[labels]
    ax.errorbar(range(4), d['estimate'], yerr=[d['estimate'] - d['lo'], d['hi'] - d['estimate']], fmt='o',
                color=tu.C['blue'], capsize=3, ms=6)
    ax.set_xticks(range(4)); ax.set_xticklabels(short); ax.set_title(ttl)
    if met in ('oe', 'slope'):
        ax.axhline(1, color=tu.C['grey'], lw=1, ls='--')
tu.save_fig(fig, 'fig4_ablation'); plt.show()

tu.finish_notebook({
    'ablation': abl.round(4).to_dict('records'),
    'domain_auroc': dom.round(4).to_dict('records'),
    'domain_top_features': {k: v.to_dict() for k, v in dom_imp.items()},
    'dissimilarity_vs_abs_ln_oe': dict(rho=round(float(rho_d), 3), p=float(p_d), k=len(jj)),
})